# ASC Phase 2 — Train RoBERTa-base

Fine-tune `roberta-base` trên 2M pseudo-labeled + gold train.
Đánh giá trên gold test set.

In [1]:
!pip install -q transformers datasets accelerate scikit-learn

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Config

In [3]:
import os

SEED = 117
MODEL_NAME = 'roberta-base'
MAX_LENGTH = 192
BATCH_SIZE = 32
GRAD_ACCUM = 2        # effective batch size = 64
EPOCHS = 3
LR = 2e-5
WARMUP_RATIO = 0.1

DRIVE = '/content/drive/MyDrive'
DATA_DIR = f'{DRIVE}/asc_phase2_data'

TRAIN_PATH = f'{DATA_DIR}/asc_phase2_train.parquet'
TEST_PATH  = f'{DATA_DIR}/gold_test_flat.parquet'

CHECKPOINT_DIR = '/content/asc_phase2_ckpt'
DRIVE_CKPT_DIR = f'{DRIVE}/asc_phase2_checkpoints'
MODEL_SAVE_DIR = f'{DRIVE}/asc_phase2_model'

LABEL_NAMES = ['negative', 'neutral', 'positive']

for name, path in [('train', TRAIN_PATH), ('test', TEST_PATH)]:
    status = 'OK' if os.path.exists(path) else 'MISSING'
    print(f'  [{status}] {name}: {path}')

  [OK] train: /content/drive/MyDrive/asc_phase2_data/asc_phase2_train.parquet
  [OK] test: /content/drive/MyDrive/asc_phase2_data/gold_test_flat.parquet


## Load data

In [4]:
import re
import numpy as np
import pandas as pd
import torch

torch.manual_seed(SEED)
np.random.seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

df_train = pd.read_parquet(TRAIN_PATH)
df_test  = pd.read_parquet(TEST_PATH)

print(f'\nTrain: {len(df_train):,}')
print(f'Test:  {len(df_test):,}')
print(f'\nTrain sentiment dist:')
print(df_train['sentiment'].value_counts().sort_index().to_string())
print(f'\nTest sentiment dist:')
print(df_test['sentiment'].value_counts().sort_index().to_string())

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB

Train: 1,999,972
Test:  782

Train sentiment dist:
sentiment
0    967540
1     66887
2    965545

Test sentiment dist:
sentiment
0    256
1     30
2    496


## Preprocessing 

- `clean_text`: collapse whitespace
- `mark_aspect`: bọc aspect bằng `[ASP]` / `[/ASP]`

In [5]:
_ws = re.compile(r'\s+')


def clean_text(text):
    if pd.isna(text):
        return ''
    return _ws.sub(' ', str(text)).strip()


def mark_aspect(sentence, aspect):
    pattern = re.compile(re.escape(aspect), flags=re.IGNORECASE)
    marked = pattern.sub(f'[ASP] {aspect} [/ASP]', sentence, count=1)
    if marked == sentence:
        marked = f'{sentence} [ASP] {aspect} [/ASP]'
    return marked


df_train['text'] = df_train.apply(
    lambda r: mark_aspect(clean_text(r['sentence_text']), clean_text(r['aspect'])),
    axis=1,
)
df_train['label'] = df_train['sentiment'].astype(int)

df_test['text'] = df_test.apply(
    lambda r: mark_aspect(clean_text(r['sentence_text']), clean_text(r['aspect'])),
    axis=1,
)
df_test['label'] = df_test['sentiment'].astype(int)

print('Sample texts:')
for i in range(3):
    row = df_train.iloc[i]
    print(f"  [{row['label']}] {row['text'][:120]}")

Sample texts:
  [0] 400 boat anchor this [ASP] printer [/ASP] started failing printing light on 12 the page with less than 3200 pages printe
  [0] the [ASP] paper [/ASP] is made of cotton and it is as others have said quite yellow and uneven ranging from yellow to br
  [0] knife [GENERIC_NOUN] [ASP] pen [/ASP] not ballpoint [GENERIC_NOUN] these pens are not ball pointed when you write with t


## Tokenize

In [6]:
from transformers import AutoTokenizer
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
special_tokens = {'additional_special_tokens': ['[ASP]', '[/ASP]']}
tokenizer.add_special_tokens(special_tokens)

train_dataset = Dataset.from_pandas(df_train[['text', 'label']], preserve_index=False)
test_dataset  = Dataset.from_pandas(df_test[['text', 'label']], preserve_index=False)


def tokenize_fn(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )


train_dataset = train_dataset.map(tokenize_fn, batched=True, batch_size=4096, remove_columns=['text'])
test_dataset  = test_dataset.map(tokenize_fn, batched=True, batch_size=4096, remove_columns=['text'])

train_dataset.set_format('torch')
test_dataset.set_format('torch')

print(f'Train dataset: {len(train_dataset):,}')
print(f'Test dataset:  {len(test_dataset):,}')
print(f'Features: {train_dataset.features}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/1999972 [00:00<?, ? examples/s]

Map:   0%|          | 0/782 [00:00<?, ? examples/s]

Train dataset: 1,999,972
Test dataset:  782
Features: {'label': Value('int64'), 'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8'))}


## Model

In [7]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
)
model.resize_token_embeddings(len(tokenizer))

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params: {n_params / 1e6:.1f}M')

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and

Trainable params: 124.6M


## Training

In [8]:
import shutil
import glob
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import TrainingArguments, Trainer, TrainerCallback


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average='macro'
    )
    p_w, r_w, f1_w, _ = precision_recall_fscore_support(
        labels, preds, average='weighted'
    )
    return {
        'accuracy': acc,
        'f1_macro': f1_macro,
        'precision_macro': p_macro,
        'recall_macro': r_macro,
        'f1_weighted': f1_w,
        'precision_weighted': p_w,
        'recall_weighted': r_w,
    }


class DriveCheckpointCallback(TrainerCallback):
    """Copy checkpoint lên Drive sau mỗi lần save (mỗi epoch)."""

    def on_save(self, args, state, control, **kwargs):
        src = os.path.join(args.output_dir, f'checkpoint-{state.global_step}')
        if not os.path.isdir(src):
            return
        dst = os.path.join(DRIVE_CKPT_DIR, f'checkpoint-{state.global_step}')
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f'\n>>> Checkpoint copied to Drive: {dst}')

        # Giữ tối đa 2 checkpoint trên Drive
        ckpts = sorted(glob.glob(os.path.join(DRIVE_CKPT_DIR, 'checkpoint-*')))
        while len(ckpts) > 2:
            shutil.rmtree(ckpts.pop(0))
            print(f'>>> Removed old Drive checkpoint')


training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    save_total_limit=2,
    fp16=True,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    logging_steps=500,
    seed=SEED,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[DriveCheckpointCallback()],
)

print(f'Steps/epoch: {len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM):,}')
print(f'Total steps:  {len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM) * EPOCHS:,}')
print(f'Checkpoints sẽ được lưu trên Drive: {DRIVE_CKPT_DIR}')

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Steps/epoch: 31,249
Total steps:  93,747
Checkpoints sẽ được lưu trên Drive: /content/drive/MyDrive/asc_phase2_checkpoints


In [9]:
# Auto-detect checkpoint từ Drive (hỗ trợ đổi tài khoản Colab)
resume_ckpt = None
drive_ckpts = sorted(glob.glob(os.path.join(DRIVE_CKPT_DIR, 'checkpoint-*')))

if drive_ckpts:
    latest_drive = drive_ckpts[-1]
    local_ckpt = os.path.join(CHECKPOINT_DIR, os.path.basename(latest_drive))

    if not os.path.exists(local_ckpt):
        print(f'Restoring checkpoint from Drive: {latest_drive}')
        os.makedirs(CHECKPOINT_DIR, exist_ok=True)
        shutil.copytree(latest_drive, local_ckpt)
        print(f'  -> Copied to: {local_ckpt}')

    resume_ckpt = local_ckpt
    print(f'Resuming from: {resume_ckpt}')
else:
    print('No checkpoint found. Training from scratch.')

trainer.train(resume_from_checkpoint=resume_ckpt)

Restoring checkpoint from Drive: /content/drive/MyDrive/asc_phase2_checkpoints/checkpoint-62500
  -> Copied to: /content/asc_phase2_ckpt/checkpoint-62500
Resuming from: /content/asc_phase2_ckpt/checkpoint-62500


There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,Precision Macro,Recall Macro,F1 Weighted,Precision Weighted,Recall Weighted
3,0.018221,0.981708,0.881074,0.662408,0.773741,0.642143,0.870754,0.871433,0.881074


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


>>> Checkpoint copied to Drive: /content/drive/MyDrive/asc_phase2_checkpoints/checkpoint-93750
>>> Removed old Drive checkpoint


There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=93750, training_loss=0.007213119135538737, metrics={'train_runtime': 7363.1999, 'train_samples_per_second': 814.852, 'train_steps_per_second': 12.732, 'total_flos': 2.2107487635850326e+17, 'train_loss': 0.007213119135538737, 'epoch': 3.0})

## Benchmark trên Gold Test Set

In [10]:
results = trainer.evaluate()

print('=' * 55)
print('  BENCHMARK RESULTS — GOLD TEST SET')
print('=' * 55)
for k, v in sorted(results.items()):
    if k.startswith('eval_') and k != 'eval_runtime' and 'per_second' not in k:
        print(f'  {k.replace("eval_", ""):25s} : {v:.4f}')

  BENCHMARK RESULTS — GOLD TEST SET
  accuracy                  : 0.8811
  f1_macro                  : 0.6624
  f1_weighted               : 0.8708
  loss                      : 0.9817
  precision_macro           : 0.7737
  precision_weighted        : 0.8714
  recall_macro              : 0.6421
  recall_weighted           : 0.8811


In [11]:
from sklearn.metrics import classification_report, confusion_matrix

preds_output = trainer.predict(test_dataset)
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = preds_output.label_ids

print('Classification Report:')
print(classification_report(y_true, y_pred, target_names=LABEL_NAMES, digits=4))

print('\nConfusion Matrix:')
cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=[f'true_{n}' for n in LABEL_NAMES],
    columns=[f'pred_{n}' for n in LABEL_NAMES],
)
print(cm_df.to_string())

Classification Report:
              precision    recall  f1-score   support

    negative     0.8482    0.8516    0.8499       256
     neutral     0.5714    0.1333    0.2162        30
    positive     0.9015    0.9415    0.9211       496

    accuracy                         0.8811       782
   macro avg     0.7737    0.6421    0.6624       782
weighted avg     0.8714    0.8811    0.8708       782


Confusion Matrix:
               pred_negative  pred_neutral  pred_positive
true_negative            218             2             36
true_neutral              11             4             15
true_positive             28             1            467


## Save model

In [12]:
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
trainer.save_model(MODEL_SAVE_DIR)
tokenizer.save_pretrained(MODEL_SAVE_DIR)
print(f'Model saved to {MODEL_SAVE_DIR}')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to /content/drive/MyDrive/asc_phase2_model
